# Plots

In [ ]:
library(readr)
library(purrr)
library(dplyr)
library(stringr)
library(fuzzyjoin)
library(ggplot2)
library(dplyr)
library(tidyverse)
library(Matrix)
library(reshape2)
library(RColorBrewer)
library(rstatix)

In [ ]:
#show all columns
options(repr.matrix.max.cols = Inf,  # show all columns
        repr.matrix.max.rows = 200)  # adjust rows as you like

#dont show col types        
options(readr.show_col_types = FALSE)

## 0. Plot parameters

In [ ]:
parent_dir = c(
"/ceph.groups/mshahbazi.grp/rsakata/EXP58/output/setB_T2/cellpose_boundary",
"/ceph.groups/mshahbazi.grp/rsakata/EXP58/output/setB_T3/cellpose_boundary",
"/ceph.groups/mshahbazi.grp/rsakata/EXP59/output/setB_T2/cellpose_boundary",
"/ceph.groups/mshahbazi.grp/rsakata/EXP60/output/setB_T3/cellpose_boundary")



sample_sheet_csv = "/ceph.groups/mshahbazi.grp/rsakata/Figures/cellboundary/sample_sheet.csv"
out_dir = "/ceph.groups/mshahbazi.grp/rsakata/Figures/cellboundary/output"

In [ ]:
if (!dir.exists(out_dir)) { 
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)
 }

In [ ]:
# define const for visualization
FONT.SIZE <- 7
LABEL.FONT.SIZE <- 7
w <- 2 
h <- 2.5
LINE.W <- 0.5/2.141959

# Set geom defaults globally
update_geom_defaults("line",      list(linewidth = LINE.W))
update_geom_defaults("errorbar",  list(linewidth = LINE.W))
#update_geom_defaults("point",     list(size = LINE.W, stroke = LINE.W))

settheme <- theme_minimal() + 
  theme(
    text = element_text(family = "sans"), 
    panel.background = element_blank(),
    panel.grid.major = element_blank(), 
    panel.grid.minor = element_blank(),
    plot.background = element_blank(),
    axis.ticks = element_line(colour = "black", linewidth = LINE.W),
    axis.ticks.length = unit(0.1, "cm"), 
    axis.line = element_line(linewidth = LINE.W, colour = "black"),
    axis.title = element_text(size = FONT.SIZE),
    axis.text = element_text(colour = "black", size = FONT.SIZE),
    strip.text = element_text(size = FONT.SIZE), 
    strip.text.y.left = element_text(angle = 0, hjust = 1, size = FONT.SIZE),
    legend.position = "right",
    legend.title = element_text(size = FONT.SIZE), 
    legend.text = element_text(size = FONT.SIZE),
    legend.key.size = unit(0.3, "cm"),
    axis.text.x = element_text(colour = "black", angle = 0, size = LABEL.FONT.SIZE),
    title = element_text(size = FONT.SIZE) 
  )

In [ ]:
col_aneu = c("euploid"= "#D4D1B3","monosomy"="#109E9D","trisomy"="#F26B3B","complex"= "#886DB0")

col_condition_2 = c("control" = "#285F63", 
               "reversine" = "#CA4F33", 
               "mosaic"= "#E2A557")

col_cell_cell = c("WT_WT" = "#5E5E5E", 
               "GFP_GFP" = "#86AB30", 
               "GFP_WT"= "#E2A557")

col_cell_cell2 = c("WT-WT" = "#5E5E5E", 
               "WT-Rev" = "#E2A557", 
               "Rev-Rev"= "#EB5951")

col_condition = c("G_R"= "#5E5E5E","Grev_R"="#86AB30","Rrev_G"="#EB5951", "Grev_Rrev"="#F0A329")

col_GFP = c("TRUE"= "#86AB30","FALSE"="#8d8d8dff")
col_state = c("Developed"= "#86AB30","Failed"="#8d8d8dff")
col_timepoint = c("D4"= "#109E9D","D6"="#4674b8")

col_mosaic =  c("-"= "#109E9D","Developed"="#4674b8", "Failed" = "#5E5E5E")


## 1. Extract summary files

In [ ]:
files <- list.files(
  path = parent_dir, 
  pattern = "_data\\.csv$", 
  recursive = TRUE, 
  full.names = TRUE
)

data_merged <- files %>%
  set_names() %>%
  map_dfr(read_csv, .id = "filename")


In [ ]:
# add columns to identify the samples and image
data_merged <- data_merged %>%
  mutate(
    EXP = str_extract(filename, "(?<=EXP).{2}"), 
    image = sub("_data\\.csv$", "", basename(filename)),
    sample = sub("_.*", "", image)
  )

head(data_merged)

In [ ]:
#merge with sample sheet
sample_sheet <- read_csv(sample_sheet_csv, show_col_types = FALSE)
sample_sheet$sample <- as.character(sample_sheet$sample)

merged_df <- data_merged %>%
  left_join(sample_sheet, by = "sample")   # keeps all rows from df1

In [ ]:
unique(data_merged$image)

In [ ]:
order_sample <- c(
  "D4_GR",  "D4_GrevRrev", "D4_GrevR", "D4_RrevG",
  "D6_GR_D", "D6_GR_F", "D6_GrevRrev_D", "D6_GrevRrev_F", 
  "D6_GrevR_D", "D6_GrevR_F", "D6_RrevG_D", "D6_RrevG_F")
merged_df <- merged_df %>%
  mutate(sample_name = factor(sample_name, levels = order_sample))

In [ ]:
tbl <- merged_df %>%
  group_by(EXP, condition, sample_name) %>%
  summarise(n_images = n_distinct(image), .groups = "drop")

tbl

## Plots

### A) Normalised intensity (mean of mean per image)

In [ ]:
# summarize the data
plot_data <- merged_df %>%
  filter(channel %in% c( "ECAD", "phalloidin")) %>%
  # mean of each image
  group_by(EXP, sample_name, condition, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE)
  ) 

In [ ]:
head(plot_data)

In [ ]:
w <- 8
h <- 2
title = "Normalised_intensity_meanperimage"
options(repr.plot.width=w, repr.plot.height=h)

p <- ggplot(plot_data, aes(x = distance_from_peak, y = mean_int_1, color = condition, group = image )) +
  
  # The main average line
  geom_line(linewidth = 0.1) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "Cell Pair Class",
    fill  = "Cell Pair Class"
  )+ facet_grid(channel~sample_name, scales = "free_y") +
  settheme +
  theme(legend.position = "none")+ 
  scale_fill_manual(values = col_condition)+ 
  scale_color_manual(values = col_condition)

ggsave(plot = p, filename = sprintf("%s/A_%s.pdf",out_dir, title), w = w, h = h)
p

### B) Background substracted normalised background 

In [ ]:
head(merged_df)

In [ ]:
# summarize the data
plot_data <- merged_df %>%
  # 1. Filter first (makes everything faster)
  filter(channel %in% c( "ECAD", "phalloidin")) %>%

  # mean of each image
  group_by(EXP, sample_name, condition, condition_2, state, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%

  # mean of each sample
  group_by(sample_name, condition, condition_2, state ,distance_from_peak, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n), # Standard Error
    .groups  = "drop"
  ) %>%

  # 3. Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(sample_name, condition, channel) %>%
  mutate(
    background = min(mean_int, na.rm = TRUE),
    corrected_mean = mean_int - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - se_int, 
    ymax = corrected_mean + se_int
  )

In [ ]:
w <- 8
h <- 2
title = "Normalised_intensity-background"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_from_peak, y = corrected_mean, fill = condition, color = condition)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "Cell Pair Class",
    fill  = "Cell Pair Class"
  )+ facet_grid( channel ~ sample_name, scales = "free_y") +
  settheme +
  theme(legend.position = "none")+ 
  scale_fill_manual(values = col_condition)+ 
  scale_color_manual(values = col_condition)

ggsave(plot = plot , filename = sprintf("%s/B_%s.pdf",out_dir, title), w = w, h = h)
plot 

In [ ]:
# summarize the data
plot_data <- merged_df %>%
  filter(channel %in% c( "ECAD", "phalloidin")) %>%

  # mean of each image
  group_by(EXP, sample_name, condition, state, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%
  # background of each image
  group_by(EXP, sample_name, condition, state, image, channel) %>%
  mutate(
    background = min(mean_int_1, na.rm = TRUE),
    corrected_mean = mean_int_1 - background
  ) %>%
  filter(distance_from_peak == 0) %>%
  ungroup() 

In [ ]:
w <- 4
h <- 2
title = "peakintensity"
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(plot_data, aes(x = sample_name, y = corrected_mean)) +  # dots for each file
  stat_summary(
    fun = mean, 
    geom = "bar",
    position = position_dodge(width = 0.75),
    fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
  geom_jitter(
    aes(color = condition),
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
    size = 0.5, alpha = 0.8
  )  +  # average bar
  stat_summary(
    fun.data = mean_se, 
    geom = "errorbar",
    position = position_dodge(width = 0.75),
    width = 0.1, 
    color = "black")+
    scale_y_continuous(limits = c(0, NA), expand = c(0, 0))+
  labs(
    title =title,
    y = "",
    x = ""
  )+ facet_wrap(~channel, nrow= 1, scales = "free_y") + settheme+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none"
  ) + 
  scale_fill_manual(values = col_condition)+ 
  scale_color_manual(values = col_condition)
              

ggsave(plot = p  , filename = sprintf("%s/B2_%s.pdf",out_dir, title), w = w, h = h)
p 

### C) by timepoint

In [ ]:
unique(merged_df$condition_2)

In [ ]:
# by time
plot_data <- merged_df %>%
  # 1. Filter first (makes everything faster)
  filter(channel %in% c( "ECAD", "phalloidin")) %>%

  # mean of each image
  group_by(sample_name, condition_2, state, timepoint, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%

  # mean of each state
  group_by(timepoint, distance_from_peak, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n), # Standard Error
    .groups  = "drop"
  ) %>%

  # 3. Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(timepoint, channel) %>%
  mutate(
    background = min(mean_int, na.rm = TRUE),
    corrected_mean = mean_int - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - se_int, 
    ymax = corrected_mean + se_int
  )

In [ ]:
w <- 2
h <- 1.5
title = "timepoint"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_from_peak, y = corrected_mean, fill = timepoint, color = timepoint)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "timepoint",
    fill  = "timepoint"
  )+ facet_grid( ~ channel, scales = "free_y") +
  settheme+ 
  scale_fill_manual(values = col_timepoint)+ 
  scale_color_manual(values = col_timepoint)

ggsave(plot = plot , filename = sprintf("%s/C_%s.pdf",out_dir, title), w = w, h = h)
plot 

In [ ]:
# summarize the data
plot_data <- merged_df %>%
  filter(channel %in% c( "ECAD", "phalloidin")) %>%

  # mean of each image
  group_by(EXP, sample_name, condition, state,timepoint, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%
  # background of each image
  group_by(EXP, sample_name, condition, state, timepoint, image, channel) %>%
  mutate(
    background = min(mean_int_1, na.rm = TRUE),
    corrected_mean = mean_int_1 - background
  ) %>%
  filter(distance_from_peak == 0) %>%
  ungroup() 

In [ ]:
w <- 2
h <- 1.5
title = "timepoint_dots"
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(plot_data, aes(x = timepoint, y = corrected_mean)) +  # dots for each file
  stat_summary(
    fun = mean, 
    geom = "bar",
    position = position_dodge(width = 0.75),
    fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
  geom_jitter(
    aes(color = timepoint),
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
    size = 0.5, alpha = 0.8
  )  +  # average bar
  stat_summary(
    fun.data = mean_se, 
    geom = "errorbar",
    position = position_dodge(width = 0.75),
    width = 0.1, 
    color = "black")+
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
  labs(
    title =title,
    y = "Normalized Intensity",
    x = ""
  )+ facet_wrap(~channel, nrow= 1, scales = "free_y") + settheme+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none"
  ) +  
  scale_fill_manual(values = col_timepoint)+ 
  scale_color_manual(values = col_timepoint)
              

ggsave(plot = p  , filename = sprintf("%s/C2_%s.pdf",out_dir, title), w = w, h = h)
p 

In [ ]:
unique(merged_df$ condition_2)

In [ ]:
# by time, only control
plot_data <- merged_df %>%
  # 1. Filter first (makes everything faster)
  filter(channel %in% c( "ECAD")) %>%
  filter(condition_2 %in% c("control")) %>%

  # mean of each image
  group_by(sample_name, condition_2, state, timepoint, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%

  # mean of each time point
  group_by(timepoint, distance_from_peak, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n), # Standard Error
    .groups  = "drop"
  ) %>%

  # 3. Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(timepoint, channel) %>%
  mutate(
    background = min(mean_int, na.rm = TRUE),
    corrected_mean = mean_int - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - se_int, 
    ymax = corrected_mean + se_int
  )

In [ ]:
w <- 2.2
h <- 1.8
title = "timepoint"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_from_peak, y = corrected_mean, fill = timepoint, color = timepoint)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "timepoint",
    fill  = "timepoint"
  )+ facet_grid( ~ channel, scales = "free_y") +
  settheme+ 
  scale_fill_manual(values = col_timepoint)+ 
  scale_color_manual(values = col_timepoint)

ggsave(plot = plot , filename = sprintf("%s/C_%s.pdf",out_dir, title), w = w, h = h)
plot 

In [ ]:
# summarize the data
plot_data <- merged_df %>%
  filter(channel %in% c( "ECAD"))  %>%
  filter(condition_2 %in% c("control"))%>%

  # mean of each image
  group_by(EXP, sample_name, condition, state,timepoint, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%
  # background of each image
  group_by(EXP, sample_name, condition, state, timepoint, image, channel) %>%
  mutate(
    background = min(mean_int_1, na.rm = TRUE),
    corrected_mean = mean_int_1 - background
  ) %>%
  filter(distance_from_peak == 0) %>%
  ungroup() 

In [ ]:
w <- 1
h <- 1.9
title = "timepoint_dots"
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(plot_data, aes(x = timepoint, y = corrected_mean)) +  # dots for each file
  stat_summary(
    fun = mean, 
    geom = "bar",
    position = position_dodge(width = 0.75),
    fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
  geom_jitter(
    aes(color = timepoint),
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
    size = 0.5, alpha = 0.8
  )  +  # average bar
  stat_summary(
    fun.data = mean_se, 
    geom = "errorbar",
    position = position_dodge(width = 0.75),
    width = 0.1, 
    color = "black")+
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
  labs(
    title =title,
    y = "Normalized Intensity",
    x = ""
  )+ facet_wrap(~channel, nrow= 1, scales = "free_y") + settheme+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none"
  ) +  
  scale_fill_manual(values = col_timepoint)+ 
  scale_color_manual(values = col_timepoint)
              

ggsave(plot = p  , filename = sprintf("%s/C2_%s.pdf",out_dir, title), w = w, h = h)
p 

In [ ]:
library(dplyr)
library(tidyr)
library(purrr)

check_test <- function(data, group_var = "timepoint", value_var = "corrected_mean",
                        conditions = c("D4", "D4"), alpha = 0.05) {

  d <- data %>%
    filter(.data[[group_var]] %in% conditions) %>%
    droplevels()

  d %>%
    group_by(state) %>%
    group_modify(~ {
      g <- split(.x[[value_var]], .x[[group_var]])
      g <- g[conditions]                       # keep the two groups in order

      # need at least 3 non-NA points per group for Shapiro
      n_ok <- all(sapply(g, function(x) sum(!is.na(x)) >= 3))

      if (!n_ok) {
        return(tibble(
          n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
          shapiro_p1 = NA_real_, shapiro_p2 = NA_real_,
          levene_p = NA_real_, normal = NA,
          recommended = "too few points (use Wilcoxon / be cautious)"
        ))
      }

      # normality per group
      sp1 <- shapiro.test(g[[1]])$p.value
      sp2 <- shapiro.test(g[[2]])$p.value
      normal <- (sp1 > alpha) & (sp2 > alpha)

      # equal-variance check (F-test; swap for car::leveneTest if preferred)
      var_p <- tryCatch(var.test(g[[1]], g[[2]])$p.value, error = function(e) NA_real_)

      rec <- if (normal) {
        if (!is.na(var_p) && var_p > alpha) "Student t-test (var.equal = TRUE)"
        else "Welch t-test"
      } else {
        "Wilcoxon rank-sum test"
      }

      tibble(
        n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
        shapiro_p1 = sp1, shapiro_p2 = sp2,
        levene_p = var_p, normal = normal,
        recommended = rec
      )
    }) %>%
    ungroup()
}

# usage
check_test(plot_data)

In [ ]:
library(rstatix)
plot_data %>%
 # group_by(condition) %>%
  t_test(
    corrected_mean ~ timepoint,
    p.adjust.method = "holm"
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

## Control developed only for day 6

In [ ]:
# by time, only control
plot_data <- merged_df %>%
  # 1. Filter first (makes everything faster)
  filter(channel %in% c( "ECAD")) %>%
  filter(condition_2 %in% c("control")) %>%
  filter(!state == "Failed")%>%

  # mean of each image
  group_by(sample_name, condition_2, state, timepoint, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%

  # mean of each time point
  group_by(timepoint, distance_from_peak, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n), # Standard Error
    .groups  = "drop"
  ) %>%

  # 3. Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(timepoint, channel) %>%
  mutate(
    background = min(mean_int, na.rm = TRUE),
    corrected_mean = mean_int - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - se_int, 
    ymax = corrected_mean + se_int
  )

In [ ]:
w <- 2.2
h <- 1.8
title = "timepoint_developed"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_from_peak, y = corrected_mean, fill = timepoint, color = timepoint)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "timepoint",
    fill  = "timepoint"
  )+ facet_grid( ~ channel, scales = "free_y") +
  settheme+ 
  scale_fill_manual(values = col_timepoint)+ 
  scale_color_manual(values = col_timepoint)

ggsave(plot = plot , filename = sprintf("%s/C_%s.pdf",out_dir, title), w = w, h = h)
plot 

In [ ]:
# summarize the data
plot_data <- merged_df %>%
  filter(channel %in% c( "ECAD"))  %>%
  filter(condition_2 %in% c("control"))%>%
  filter(!state == "Failed")%>%

  # mean of each image
  group_by(EXP, sample_name, condition, state,timepoint, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%
  # background of each image
  group_by(EXP, sample_name, condition, state, timepoint, image, channel) %>%
  mutate(
    background = min(mean_int_1, na.rm = TRUE),
    corrected_mean = mean_int_1 - background
  ) %>%
  filter(distance_from_peak == 0) %>%
  ungroup() 

In [ ]:
w <- 1
h <- 1.9
title = "timepoint_dots_developed"
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(plot_data, aes(x = timepoint, y = corrected_mean)) +  # dots for each file
  stat_summary(
    fun = mean, 
    geom = "bar",
    position = position_dodge(width = 0.75),
    fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
  geom_jitter(
    aes(color = timepoint),
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
    size = 0.5, alpha = 0.8
  )  +  # average bar
  stat_summary(
    fun.data = mean_se, 
    geom = "errorbar",
    position = position_dodge(width = 0.75),
    width = 0.1, 
    color = "black")+
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
  labs(
    title =title,
    y = "Normalized Intensity",
    x = ""
  )+ facet_wrap(~channel, nrow= 1, scales = "free_y") + settheme+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none"
  ) +  
  scale_fill_manual(values = col_timepoint)+ 
  scale_color_manual(values = col_timepoint)
              

ggsave(plot = p  , filename = sprintf("%s/C2_%s.pdf",out_dir, title), w = w, h = h)
p 

In [ ]:
library(dplyr)
library(tidyr)
library(purrr)

check_test <- function(data, group_var = "timepoint", value_var = "corrected_mean",
                        conditions = c("D4", "D4"), alpha = 0.05) {

  d <- data %>%
    filter(.data[[group_var]] %in% conditions) %>%
    droplevels()

  d %>%
    group_by(state) %>%
    group_modify(~ {
      g <- split(.x[[value_var]], .x[[group_var]])
      g <- g[conditions]                       # keep the two groups in order

      # need at least 3 non-NA points per group for Shapiro
      n_ok <- all(sapply(g, function(x) sum(!is.na(x)) >= 3))

      if (!n_ok) {
        return(tibble(
          n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
          shapiro_p1 = NA_real_, shapiro_p2 = NA_real_,
          levene_p = NA_real_, normal = NA,
          recommended = "too few points (use Wilcoxon / be cautious)"
        ))
      }

      # normality per group
      sp1 <- shapiro.test(g[[1]])$p.value
      sp2 <- shapiro.test(g[[2]])$p.value
      normal <- (sp1 > alpha) & (sp2 > alpha)

      # equal-variance check (F-test; swap for car::leveneTest if preferred)
      var_p <- tryCatch(var.test(g[[1]], g[[2]])$p.value, error = function(e) NA_real_)

      rec <- if (normal) {
        if (!is.na(var_p) && var_p > alpha) "Student t-test (var.equal = TRUE)"
        else "Welch t-test"
      } else {
        "Wilcoxon rank-sum test"
      }

      tibble(
        n1 = sum(!is.na(g[[1]])), n2 = sum(!is.na(g[[2]])),
        shapiro_p1 = sp1, shapiro_p2 = sp2,
        levene_p = var_p, normal = normal,
        recommended = rec
      )
    }) %>%
    ungroup()
}

# usage
check_test(plot_data)

In [ ]:
library(rstatix)
plot_data %>%
 # group_by(condition) %>%
  t_test(
    corrected_mean ~ timepoint,
    p.adjust.method = "holm"
  ) %>%
  mutate(
    p_use = if ("p.adj" %in% names(.)) p.adj else p,   # fall back to raw p
    stars = case_when(
      p_use < 0.0001 ~ "****",
      p_use < 0.001  ~ "***",
      p_use < 0.01   ~ "**",
      p_use < 0.05   ~ "*",
      TRUE           ~ "ns"
    )
  )

### Mosaic - condition and timepoint

In [ ]:
head(merged_df)

In [ ]:
unique(merged_df$state	)

In [ ]:
# by time, only control
plot_data <- merged_df %>%
  # 1. Filter firs (makes everything faster)
  filter(channel %in% c( "ECAD")) %>%
  filter(condition_2 %in% c("mosaic")) %>%

  # mean of each image
  group_by(sample_name, condition_2, state, timepoint, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%

  # mean of each time point
  group_by(state, distance_from_peak, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n), # Standard Error
    .groups  = "drop"
  ) %>%

  # 3. Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(state, channel) %>%
  mutate(
    background = min(mean_int, na.rm = TRUE),
    corrected_mean = mean_int - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - se_int, 
    ymax = corrected_mean + se_int
  )

In [ ]:
w <- 2.5
h <- 1.8
title = "mosaic"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_from_peak, y = corrected_mean, fill = state, color = state)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "timepoint",
    fill  = "timepoint"
  )+ facet_grid( ~ channel, scales = "free_y") +
  settheme+ 
  scale_fill_manual(values = col_mosaic)+ 
  scale_color_manual(values = col_mosaic)

ggsave(plot = plot , filename = sprintf("%s/C_%s.pdf",out_dir, title), w = w, h = h)
plot 

In [ ]:
# summarize the data
plot_data <- merged_df %>%
  filter(channel %in% c( "ECAD"))  %>%
  filter(condition_2 %in% c("mosaic"))%>%

  # mean of each image
  group_by(EXP, sample_name, condition, state,timepoint, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%
  # background of each image
  group_by(EXP, sample_name, condition, state, timepoint, image, channel) %>%
  mutate(
    background = min(mean_int_1, na.rm = TRUE),
    corrected_mean = mean_int_1 - background
  ) %>%
  filter(distance_from_peak == 0) %>%
  ungroup() 

In [ ]:
w <- 1.2
h <- 2
title = "mosaic_dots"
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(plot_data, aes(x = state, y = corrected_mean)) +  # dots for each file
  stat_summary(
    fun = mean, 
    geom = "bar",
    position = position_dodge(width = 0.75),
    fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
  geom_jitter(
    aes(color = state),
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
    size = 0.5, alpha = 0.8
  )  +  # average bar
  stat_summary(
    fun.data = mean_se, 
    geom = "errorbar",
    position = position_dodge(width = 0.75),
    width = 0.1, 
    color = "black")+
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
  labs(
    title =title,
    y = "Normalized Intensity",
    x = ""
  )+ facet_wrap(~channel, nrow= 1, scales = "free_y") + settheme+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none"
  ) +  
  scale_fill_manual(values = col_mosaic)+ 
  scale_color_manual(values = col_mosaic)
              

ggsave(plot = p  , filename = sprintf("%s/C2_%s.pdf",out_dir, title), w = w, h = h)
p 

In [ ]:
anova_res <- plot_data %>%
anova_test(corrected_mean ~ state) %>%
add_significance(p.col = "p")

anova_res

In [ ]:
n_per_state <- plot_data %>%
  count(state, name = "n")

n_per_state

### D) by condition

In [ ]:
# by condition
plot_data <- merged_df %>%
  # 1. Filter first (makes everything faster)
  filter(channel %in% c( "ECAD", "phalloidin")) %>%

  filter(timepoint %in% c( "D4"))%>%

  # mean of each image
  group_by(sample_name, condition_2, state, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%

  # mean of each state
  group_by(condition_2, distance_from_peak, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n), # Standard Error
    .groups  = "drop"
  ) %>%

  # 3. Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(condition_2, channel) %>%
  mutate(
    background = min(mean_int, na.rm = TRUE),
    corrected_mean = mean_int - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - se_int, 
    ymax = corrected_mean + se_int
  )

In [ ]:
w <- 3.5
h <- 1.5
title = "condition"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_from_peak, y = corrected_mean, fill = condition_2, color = condition_2)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "condition_2",
    fill  = "condition_2"
  )+ facet_grid( ~ channel, scales = "free_y") +
  settheme + 
  scale_fill_manual(values = col_condition_2)+ 
  scale_color_manual(values = col_condition_2)

ggsave(plot = plot , filename = sprintf("%s/D_%s.pdf",out_dir, title), w = w, h = h)
plot 

In [ ]:
# summarize the data
plot_data <- merged_df %>%
  filter(channel %in% c( "ECAD", "phalloidin")) %>%

  # mean of each image
  group_by(EXP, sample_name, condition, condition_2, state,timepoint, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%
  # background of each image
  group_by(EXP, sample_name, condition, condition_2, state, timepoint, image, channel) %>%
  mutate(
    background = min(mean_int_1, na.rm = TRUE),
    corrected_mean = mean_int_1 - background
  ) %>%
  filter(distance_from_peak == 0) %>%
  ungroup() 

In [ ]:
w <- 2.2
h <- 1.8
title = "condition_dots"
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(plot_data, aes(x = condition_2, y = corrected_mean)) +  # dots for each file
  stat_summary(
    fun = mean, 
    geom = "bar",
    position = position_dodge(width = 0.75),
    fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
  geom_jitter(
    aes(color = condition_2),
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
    size = 0.5, alpha = 0.8
  )  +  # average bar
  stat_summary(
    fun.data = mean_se, 
    geom = "errorbar",
    position = position_dodge(width = 0.75),
    width = 0.1, 
    color = "black")+
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
  labs(
    title =title,
    y = "Normalized Intensity",
    x = ""
  )+ facet_wrap(~channel, nrow= 1, scales = "free_y") + settheme+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none"
  ) +  
  scale_fill_manual(values = col_condition_2)+ 
  scale_color_manual(values = col_condition_2)
              

ggsave(plot = p  , filename = sprintf("%s/D2_%s.pdf",out_dir, title), w = w, h = h)
p 

### E) Failed vs developed

In [ ]:
# failed vs developed
plot_data <- merged_df %>%
  filter(channel %in% c( "ECAD", "phalloidin"))%>%
  filter(timepoint %in% c( "D6")) %>%

  # mean of each image
  group_by(sample_name, state, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%

  # mean of each state
  group_by(state ,distance_from_peak, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n), # Standard Error
    .groups  = "drop"
  ) %>%

  # 3. Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(state, channel) %>%
  mutate(
    background = min(mean_int, na.rm = TRUE),
    corrected_mean = mean_int - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - se_int, 
    ymax = corrected_mean + se_int
  )

In [ ]:
w <- 3.5
h <- 1.5
title = "failed_developed"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_from_peak, y = corrected_mean, fill = state, color = state)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "state",
    fill  = "state"
  )+ facet_grid( ~ channel, scales = "free_y") +
  settheme + 
  scale_fill_manual(values = col_state)+ 
  scale_color_manual(values = col_state)

ggsave(plot = plot , filename = sprintf("%s/E_%s.pdf",out_dir, title), w = w, h = h)
plot 

In [ ]:
# summarize the data
plot_data <- merged_df %>%
  filter(channel %in% c( "ECAD", "phalloidin")) %>%
  filter(timepoint %in% c( "D6")) %>%

  # mean of each image
  group_by(EXP, sample_name, condition, state,timepoint, image, distance_from_peak, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%
  # background of each image
  group_by(EXP, sample_name, condition, state, timepoint, image, channel) %>%
  mutate(
    background = min(mean_int_1, na.rm = TRUE),
    corrected_mean = mean_int_1 - background
  ) %>%
  filter(distance_from_peak == 0) %>%
  ungroup() 

In [ ]:
w <- 2
h <- 1.8
title = "state_dots"
options(repr.plot.width=w, repr.plot.height=h)

p = ggplot(plot_data, aes(x = state, y = corrected_mean)) +  # dots for each file
  stat_summary(
    fun = mean, 
    geom = "bar",
    position = position_dodge(width = 0.75),
    fill = "grey", alpha = 0.4, width = 0.6) +   # error bars
  geom_jitter(
    aes(color = state),
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.75),
    size = 0.5, alpha = 0.8
  )  +  # average bar
  stat_summary(
    fun.data = mean_se, 
    geom = "errorbar",
    position = position_dodge(width = 0.75),
    width = 0.1, 
    color = "black")+
    scale_y_continuous(limits = c(0, NA), 
    expand = expansion(mult = c(0, 0.1)))+
  labs(
    title =title,
    y = "Normalized Intensity",
    x = ""
  )+ facet_wrap(~channel, nrow= 1, scales = "free_y") + settheme+
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1),
    legend.position = "none"
  ) +  
  scale_fill_manual(values = col_state)+ 
  scale_color_manual(values = col_state)
              

ggsave(plot = p  , filename = sprintf("%s/E2_%s.pdf",out_dir, title), w = w, h = h)
p 

### F) Cell-Cell: GFP intensity

In [ ]:
gfp_df <- merged_df %>% 
  filter(channel == "GFP")


pair_stats <- gfp_df %>%
  group_by(EXP, sample_name, condition, image, z, label_i, label_j) %>%
  summarise(
    intensity_before = mean(normalized_intensity[distance_from_peak < 0], na.rm = TRUE),
    intensity_after  = mean(normalized_intensity[distance_from_peak > 0], na.rm = TRUE),
    .groups = "drop"
    )

plot_data <- pair_stats %>%
  pivot_longer(cols = c(intensity_before, intensity_after), 
                names_to = "position", values_to = "intensity") 

In [ ]:
w <- 3
h <- 8
title = "gfp_intensity"
options(repr.plot.width=w, repr.plot.height=h)

GFP_thresh = 1


  p <- ggplot(plot_data, aes(x = intensity, fill = condition)) +
    geom_histogram(bins = 50, alpha = 0.5, position = "identity") +
    #scale_fill_manual(values = c(intensity_before = "blue", intensity_after = "orange")) +
    labs(title = "GFP Intensity Distribution", fill = "Group") +
    facet_wrap(~sample_name, ncol=1, scales = "free_y")+ 
    geom_vline(xintercept = GFP_thresh,
              linetype = "dashed")+ 
  scale_fill_manual(values = col_condition)+ 
  scale_color_manual(values = col_condition)+ settheme +
  theme(legend.position = "none")
              
  
ggsave(plot = p , filename = sprintf("%s/f_%s.pdf",out_dir, title), w = w, h = h)
p

In [ ]:
pair_stats <- pair_stats %>%
  mutate(
    cell_cell = case_when(
      intensity_before < GFP_thresh & intensity_after < GFP_thresh ~ "WT_WT",
      intensity_before > GFP_thresh & intensity_after > GFP_thresh ~ "GFP_GFP",
      intensity_before == "NaN" | intensity_after == "NaN" ~ NA, 
      TRUE ~ "GFP_WT" # Default case (mix of high/low)
    )
  )


In [ ]:
head(pair_stats)

In [ ]:
  # --- STEP 4: Merge back to original data ---
  df_final <- merged_df %>%
    select(-cell_cell)%>%
    left_join(pair_stats %>% select(EXP, sample_name,image, z, label_i, label_j, cell_cell), 
              by = c("EXP", "sample_name" , "image", "z", "label_i", "label_j"))


In [ ]:
head(df_final)

In [ ]:
df_final <- df_final%>%
  mutate(
    cell_cell2 = case_when(
      # Condition 1: G_R
      condition == "G_R" ~ "WT-WT",
      
      # Condition 2: Grev_Rrev
      condition == "Grev_Rrev" ~ "Rev-Rev",
      
      # Condition 3: Grev_R logic
      condition == "Grev_R" & cell_cell == "WT_WT"  ~ "WT-WT",
      condition == "Grev_R" & cell_cell == "GFP_GFP" ~ "Rev-Rev",
      condition == "Grev_R" & cell_cell == "GFP_WT"  ~ "WT-Rev",
      
      # Condition 4: Rrev_G logic 
      # (Note: Fixed the dash to underscore for consistency with your examples)
      condition == "Rrev_G" & cell_cell == "WT_WT"  ~ "Rev-Rev",
      condition == "Rrev_G" & cell_cell == "GFP_GFP" ~ "WT-WT",
      condition == "Rrev_G" & cell_cell == "GFP_WT"  ~ "WT-Rev",
      
      # Default catch-all
      TRUE ~ NA_character_
    )
  )

In [ ]:
# by condition and state
plot_data <- df_final %>%
  filter(channel %in% c( "ECAD", "phalloidin")) %>%

  # mean of each image at each cell-cell group
  group_by(EXP, sample_name, condition, state, timepoint, image, distance_from_peak,  cell_cell, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%

  # mean of each sample at each cell-cell group
  group_by(sample_name ,condition, state, timepoint, distance_from_peak, cell_cell, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n),
    .groups  = "drop"
  ) %>%

  # 3. Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(sample_name, condition, state,timepoint,  cell_cell,  channel) %>%
  mutate(
    background = min(mean_int, na.rm = TRUE),
    corrected_mean = mean_int - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - se_int, 
    ymax = corrected_mean + se_int
  ) %>%
  filter(!is.na(cell_cell))

In [ ]:
w <- 4
h <- 2
title = "cell_cell_developed"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data%>%filter(state=="Developed"), aes(x = distance_from_peak, y = corrected_mean, fill = condition, color = condition)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "state",
    fill  = "state"
  )+ facet_grid(  channel~ cell_cell, scales = "free_y") +
  settheme+
  scale_fill_manual(values = col_condition)+ 
  scale_color_manual(values = col_condition)

ggsave(plot = plot , filename = sprintf("%s/F1_%s.pdf",out_dir, title), w = w, h = h)
plot 

In [ ]:
w <- 4
h <- 2
title = "cell_cell_failed"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data%>%filter(state=="Failed"), aes(x = distance_from_peak, y = corrected_mean, fill = condition, color = condition)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "state",
    fill  = "state"
  )+ facet_grid(  channel~ cell_cell, scales = "free_y") +
  settheme+
  scale_fill_manual(values = col_condition)+ 
  scale_color_manual(values = col_condition)

ggsave(plot = plot , filename = sprintf("%s/F2_%s.pdf",out_dir, title), w = w, h = h)
plot 

In [ ]:
w <- 4
h <- 2
title = "cell_cell_D4"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data%>%filter(timepoint=="D4"), aes(x = distance_from_peak, y = corrected_mean, fill = condition, color = condition)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "state",
    fill  = "state"
  )+ facet_grid(  channel~ cell_cell, scales = "free_y") +
  settheme+
  scale_fill_manual(values = col_condition)+ 
  scale_color_manual(values = col_condition)

ggsave(plot = plot , filename = sprintf("%s/F3_%s.pdf",out_dir, title), w = w, h = h)
plot 

### G) Cell-Cell: Rev / WT

In [ ]:
# by condition and state
plot_data <- df_final %>%
  filter(channel %in% c( "ECAD", "phalloidin")) %>%

  # mean of each image at each cell-cell group
  group_by(EXP, sample_name, condition, state, timepoint, image, distance_from_peak,  cell_cell2, channel) %>%
  summarise(
    mean_int_1 = mean(normalized_intensity, na.rm = TRUE), 
    .groups  = "drop"
  ) %>%

  # mean of each sample at each cell-cell group
  group_by(state, timepoint, distance_from_peak, cell_cell2, channel) %>%
  summarise(
    mean_int = mean(mean_int_1, na.rm = TRUE),
    sd_int   = sd(mean_int_1, na.rm = TRUE),
    n        = n(),
    se_int   = sd_int / sqrt(n),
    .groups  = "drop"
  ) %>%

  # 3. Calculate Background (Minimum of the mean profile)
  # We group by sample and channel (ignoring distance) to find the lowest point of the line
  group_by(state, timepoint,  cell_cell2,  channel) %>%
  mutate(
    background = min(mean_int, na.rm = TRUE),
    corrected_mean = mean_int - background
  ) %>%
  ungroup() %>%

  # 4. Define Limits for plotting
  # Note: Substracting a constant background does NOT change the SD/SE width, 
  # so we apply the error bars to the NEW corrected mean.
  mutate(
    ymin = corrected_mean - se_int, 
    ymax = corrected_mean + se_int
  ) %>%
  filter(!is.na(cell_cell2))

In [ ]:
w <- 4
h <- 2
title = "cell_cell2"
options(repr.plot.width=w, repr.plot.height=h)

plot <- ggplot(plot_data, aes(x = distance_from_peak, y = corrected_mean, fill = cell_cell2, color = cell_cell2)) +
  
  # Add the vertical dashed line at 0 (center)
  geom_vline(xintercept = 0, linetype = "dashed", color = "gray50", linewidth = 0.5) +
  
  # The shaded error band (Ribbon)
  geom_ribbon(aes(ymin = ymin, ymax = ymax), alpha = 0.2, color = NA) +
  
  # The main average line
  geom_line(linewidth = 0.5) +
  
  # Labels and Theme
  labs(
    title = title ,
    x = "Distance from Peak",
    y = "Normalized Intensity",
    color = "state",
    fill  = "state"
  )+ facet_grid(  channel~ state, scales = "free_y") +
  settheme+
  scale_fill_manual(values = col_cell_cell2)+ 
  scale_color_manual(values = col_cell_cell2)

ggsave(plot = plot , filename = sprintf("%s/G_%s.pdf",out_dir, title), w = w, h = h)
plot 